# Reading the runs back: latency, cost and prose, per part

The compiler notebook ends by recording each run. This one starts there. It is
the half of the loop that people skip, and skipping it is why so many harness
decisions are argued rather than measured.

**Three questions, three audiences:**

| Question | Who asks | What answers it |
|---|---|---|
| Where did the time and the money go? | you, next sprint | per-call observations, grouped by part and by kind |
| Is configuration B better than A? | you, choosing | the same case set under both, compared per part |
| Is the *text* any good? | the SME who owns the rules | the assembled document, as Markdown and Word |

The third one is the reason this notebook writes documents and not only tables.
A rule owner does not read traces. They read the release note, say "that third
bullet is wrong", and the only useful next question is *which model, which
surface, which skill version produced it* — which the run recorded, and which
this notebook puts next to the prose.

**What it expects in a trace.** Exactly what the compiler notebook's
`record_run` writes:

- a trace per document, named `release_notes.run`, with a `session_id` per
  experiment and tags for the experiment, the skill variant and the models;
- a `part.<name>` span per output part, whose output is the text that won and
  whose metadata is that part's provenance;
- one generation observation per model call, carrying the model, the usage and
  the part it served.

**No Langfuse credentials?** The notebook falls back to a small recorded sample
so every table and every export below still runs. It says so when it does.


## 1. Setup


In [1]:
%pip install -q "langfuse>=4,<5" python-docx pandas


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [2]:
import getpass, io, json, os, re, statistics, zipfile
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Dict, List, Optional, Sequence

import pandas as pd

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)

# Keys are read from the environment first; the prompt never echoes them.
for var in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"):
    if not os.getenv(var):
        entered = getpass.getpass(f"{var} (leave empty to use the recorded sample): ")
        if entered:
            os.environ[var] = entered
os.environ.setdefault("LANGFUSE_HOST", "https://cloud.langfuse.com")   # US: https://us.cloud.langfuse.com

# --- what to read ----------------------------------------------------------
TRACE_NAME  = "release_notes.run"
SESSION_IDS: List[str] = []      # e.g. ["ch04-1763040000"]; empty = every session in the window
TAGS: List[str]        = []      # e.g. ["experiment:split+routed"]
HOURS_BACK  = 24
MAX_TRACES  = 300                # a sweep is a few hundred traces, not fifty
ANONYMISE   = False              # True: no model names, no costs — for SME review

print(f"reading `{TRACE_NAME}` traces from the last {HOURS_BACK}h"
      + (f", sessions {SESSION_IDS}" if SESSION_IDS else "")
      + (f", tags {TAGS}" if TAGS else ""))


reading `release_notes.run` traces from the last 24h


## 2. Fetching traces for the sweep

`trace.list` returns up to 100 traces per page. When we compare two configurations on the same set of cases, the first page can contain only the newer runs. If we stop there, it can look like the baseline never ran. Fetch all pages before comparing the two configurations.

The list response gives us the trace IDs. We then fetch each trace by ID to get its observations. That’s where we can see which parts ran, which models they used, and how many tokens they consumed.

In [3]:
def _client():
    if not (os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY")):
        return None
    try:
        from langfuse import get_client
        client = get_client()
        return client if client.auth_check() else None
    except Exception as exc:
        print(f"Langfuse unavailable ({type(exc).__name__}: {str(exc)[:80]})")
        return None


def fetch_traces() -> List[object]:
    """Every matching trace in the window, then each one in full."""
    client = _client()
    if client is None:
        return []
    since = datetime.now(timezone.utc) - timedelta(hours=HOURS_BACK)

    def page_through(**kw):
        page, heads = 1, []
        while len(heads) < MAX_TRACES:
            take = min(100, MAX_TRACES - len(heads))
            batch = client.api.trace.list(name=TRACE_NAME, from_timestamp=since,
                                          limit=take, page=page, **kw).data
            heads += batch
            if len(batch) < take:
                break
            page += 1
        return heads

    if SESSION_IDS:
        heads = [t for sid in SESSION_IDS for t in page_through(session_id=sid)]
    elif TAGS:
        heads = page_through(tags=TAGS)
    else:
        heads = page_through()

    seen, unique = set(), []
    for t in heads:                       # a trace can match twice via tags
        if t.id not in seen:
            seen.add(t.id)
            unique.append(t)
    return [client.api.trace.get(t.id) for t in unique]


TRACES = fetch_traces()
print(f"{len(TRACES)} trace(s) fetched")
for t in TRACES[:8]:
    print(f"   {t.timestamp:%Y-%m-%d %H:%M}  {t.session_id}  "
          f"{(t.latency or 0):>6.1f}s  ${(t.total_cost or 0):.4f}")


0 trace(s) fetched


## 3. Parsing one run

We turn each trace into one row for the sweep comparison. To reconstruct the document, we read the `part.*` spans because each part is generated, judged, and repaired separately. We use the root span’s output to fill in fields the part spans don’t carry.

If a part fails and the final document never assembles, we can still record the parts that completed and see which one failed.

In [4]:
def _data(value):
    """Langfuse returns JSON as a dict or as a string, depending on the endpoint."""
    if isinstance(value, str):
        try:
            return json.loads(value)
        except Exception:
            return {}
    return value if isinstance(value, dict) else {}


@dataclass
class Call:
    label: str
    kind: str          # generate | judge | repair | other
    part: str
    model: str
    seconds: float
    tokens_in: int
    tokens_out: int
    cost: float


@dataclass
class Run:
    run_id: str
    started: datetime
    experiment: str
    skill: str
    ready: bool
    parts: Dict[str, object]                    # part -> the text that won
    provenance: Dict[str, Dict[str, object]]    # part -> model, surface, status, score
    calls: List[Call]
    seconds: float
    cost: float

    @property
    def tokens(self):
        return sum(c.tokens_in for c in self.calls), sum(c.tokens_out for c in self.calls)


def _kind(label: str) -> str:
    for prefix in ("split", "generate", "judge", "repair"):
        if label.startswith(prefix):
            return "generate" if prefix == "split" else prefix
    return "other"


def parse_run(trace) -> Run:
    meta = _data(getattr(trace, "metadata", None))
    root_out = _data(getattr(trace, "output", None))
    parts, provenance, calls = {}, {}, []

    for o in getattr(trace, "observations", []) or []:
        name = o.name or ""
        if name.startswith("part."):
            part = name.split(".", 1)[1]
            parts[part] = _data(getattr(o, "output", None)).get("text")
            provenance[part] = _data(getattr(o, "metadata", None))
            continue
        if o.type != "GENERATION":
            continue
        usage = o.usage_details or {}
        om = _data(getattr(o, "metadata", None))
        calls.append(Call(
            label=name, kind=_kind(name), part=om.get("part", ""), model=o.model or "",
            seconds=round(om.get("seconds") or ((o.end_time - o.start_time).total_seconds()
                          if o.end_time and o.start_time else 0.0), 2),
            tokens_in=usage.get("input") or 0, tokens_out=usage.get("output") or 0,
            cost=round(o.calculated_total_cost or 0.0, 6)))

    # The root output fills only what the parts did not carry.
    for key, value in (root_out.get("document") or {}).items():
        parts.setdefault(key, value)

    return Run(run_id=trace.session_id or trace.id, started=trace.timestamp,
               experiment=meta.get("experiment", "unlabelled"),
               skill=(_data(getattr(trace, "input", None)).get("skill")
                      or meta.get("skill", "")),
               ready=bool(root_out.get("ready")), parts=parts, provenance=provenance,
               calls=calls, seconds=round(trace.latency or 0.0, 1),
               cost=round(trace.total_cost or 0.0, 5))


In [5]:
# A recorded pair of runs, so the rest of the notebook works without credentials.
SAMPLE = json.loads(r"""
{"runs": [
 {"run_id": "sample-whole", "experiment": "whole document", "skill": "formal-developer",
  "ready": true, "seconds": 11.4,
  "parts": {"summary": "Version 2.4.0 rebuilds the CSV exporter and changes `/v1/export` to return a job id instead of the file.",
            "highlights": ["Improved export throughput, approximately two times",
                           "Reduced peak memory during export by approximately 40 percent",
                           "Revised `/v1/export` response: `202` with a job id, polled at `/v1/export/{id}`"],
            "breaking": true},
  "provenance": {"summary": {"model": "openai/gpt-5.6-luna", "surface_digest": "a1b2c3d4e5f6", "status": "ok", "score": 0.72, "attempts": 0},
                 "highlights": {"model": "openai/gpt-5.6-luna", "surface_digest": "a1b2c3d4e5f6", "status": "repaired", "score": 0.66, "attempts": 1}},
  "calls": [{"label": "generate[0]", "part": "", "model": "openai/gpt-5.6-luna", "seconds": 6.2, "tokens_in": 951, "tokens_out": 168, "cost": 0.00212},
            {"label": "generate[1]", "part": "", "model": "openai/gpt-5.6-luna", "seconds": 6.9, "tokens_in": 951, "tokens_out": 174, "cost": 0.00216},
            {"label": "generate[2]", "part": "", "model": "openai/gpt-5.6-luna", "seconds": 7.1, "tokens_in": 951, "tokens_out": 181, "cost": 0.00219},
            {"label": "judge:anthropic", "part": "summary", "model": "anthropic/claude-haiku-4.5", "seconds": 3.1, "tokens_in": 1220, "tokens_out": 210, "cost": 0.00151},
            {"label": "judge:openai", "part": "summary", "model": "openai/gpt-5.4-mini", "seconds": 2.4, "tokens_in": 1220, "tokens_out": 198, "cost": 0.00042},
            {"label": "judge:google", "part": "summary", "model": "google/gemini-3.1-flash-lite", "seconds": 1.9, "tokens_in": 1220, "tokens_out": 205, "cost": 0.00018},
            {"label": "judge:anthropic", "part": "highlights", "model": "anthropic/claude-haiku-4.5", "seconds": 3.4, "tokens_in": 1380, "tokens_out": 240, "cost": 0.00172},
            {"label": "judge:openai", "part": "highlights", "model": "openai/gpt-5.4-mini", "seconds": 2.6, "tokens_in": 1380, "tokens_out": 232, "cost": 0.00048},
            {"label": "judge:google", "part": "highlights", "model": "google/gemini-3.1-flash-lite", "seconds": 2.0, "tokens_in": 1380, "tokens_out": 228, "cost": 0.00020},
            {"label": "repair:highlights", "part": "highlights", "model": "openai/gpt-5.6-luna", "seconds": 4.8, "tokens_in": 1010, "tokens_out": 96, "cost": 0.00181}]},
 {"run_id": "sample-split", "experiment": "split+routed", "skill": "formal-developer",
  "ready": true, "seconds": 7.9,
  "parts": {"summary": "Version 2.4.0 rebuilds the CSV exporter; `/v1/export` now returns a job id rather than the file.",
            "highlights": ["Improved export throughput, approximately two times",
                           "Reduced peak memory during export by approximately 40 percent",
                           "Corrected escaping of quotation marks inside CSV fields",
                           "Revised `/v1/export` response: `202` with a job id, polled at `/v1/export/{id}`"],
            "breaking": true},
  "provenance": {"summary": {"model": "openai/gpt-5.4-mini", "surface_digest": "77aa11bb22cc", "status": "ok", "score": 0.78, "attempts": 0},
                 "highlights": {"model": "openai/gpt-5.6-luna", "surface_digest": "9f8e7d6c5b4a", "status": "ok", "score": 0.81, "attempts": 0}},
  "calls": [{"label": "split:summary[0]", "part": "summary", "model": "openai/gpt-5.4-mini", "seconds": 1.8, "tokens_in": 402, "tokens_out": 38, "cost": 0.00014},
            {"label": "split:summary[1]", "part": "summary", "model": "openai/gpt-5.4-mini", "seconds": 2.0, "tokens_in": 402, "tokens_out": 41, "cost": 0.00015},
            {"label": "split:summary[2]", "part": "summary", "model": "openai/gpt-5.4-mini", "seconds": 1.7, "tokens_in": 402, "tokens_out": 36, "cost": 0.00014},
            {"label": "split:highlights[0]", "part": "highlights", "model": "openai/gpt-5.6-luna", "seconds": 3.9, "tokens_in": 458, "tokens_out": 121, "cost": 0.00132},
            {"label": "split:highlights[1]", "part": "highlights", "model": "openai/gpt-5.6-luna", "seconds": 4.2, "tokens_in": 458, "tokens_out": 128, "cost": 0.00136},
            {"label": "split:highlights[2]", "part": "highlights", "model": "openai/gpt-5.6-luna", "seconds": 4.0, "tokens_in": 458, "tokens_out": 119, "cost": 0.00131},
            {"label": "judge:anthropic", "part": "summary", "model": "anthropic/claude-haiku-4.5", "seconds": 3.0, "tokens_in": 1190, "tokens_out": 204, "cost": 0.00148},
            {"label": "judge:openai", "part": "summary", "model": "openai/gpt-5.4-mini", "seconds": 2.3, "tokens_in": 1190, "tokens_out": 196, "cost": 0.00041},
            {"label": "judge:google", "part": "summary", "model": "google/gemini-3.1-flash-lite", "seconds": 1.8, "tokens_in": 1190, "tokens_out": 201, "cost": 0.00017},
            {"label": "judge:anthropic", "part": "highlights", "model": "anthropic/claude-haiku-4.5", "seconds": 3.3, "tokens_in": 1410, "tokens_out": 244, "cost": 0.00175},
            {"label": "judge:openai", "part": "highlights", "model": "openai/gpt-5.4-mini", "seconds": 2.5, "tokens_in": 1410, "tokens_out": 236, "cost": 0.00049},
            {"label": "judge:google", "part": "highlights", "model": "google/gemini-3.1-flash-lite", "seconds": 2.1, "tokens_in": 1410, "tokens_out": 230, "cost": 0.00021}]}
]}
""")


def sample_runs() -> List[Run]:
    runs = []
    for r in SAMPLE["runs"]:
        calls = [Call(kind=_kind(c["label"]), **c) for c in r["calls"]]
        runs.append(Run(run_id=r["run_id"], started=datetime.now(timezone.utc),
                        experiment=r["experiment"], skill=r["skill"], ready=r["ready"],
                        parts=r["parts"], provenance=r["provenance"], calls=calls,
                        seconds=r["seconds"],
                        cost=round(sum(c.cost for c in calls), 5)))
    return runs


RUNS = [parse_run(t) for t in TRACES] or sample_runs()
SOURCE = "Langfuse" if TRACES else "the recorded sample (no credentials, or no traces in the window)"
print(f"{len(RUNS)} run(s) from {SOURCE}\n")
print(f"{'run':<22}{'experiment':<18}{'skill':<20}{'ready':<7}{'calls':>6}{'sec':>7}{'USD':>9}")
for r in RUNS:
    print(f"{r.run_id:<22}{r.experiment:<18}{r.skill:<20}{str(r.ready):<7}"
          f"{len(r.calls):>6}{r.seconds:>7.1f}{r.cost:>9.4f}")


2 run(s) from the recorded sample (no credentials, or no traces in the window)

run                   experiment        skill               ready   calls    sec      USD
sample-whole          whole document    formal-developer    True       10   11.4   0.0128
sample-split          split+routed      formal-developer    True       12    7.9   0.0089


## 4. Breaking down latency and cost

A run can take `11.4s` overall, but we still need the span timings to see which part was slow. The same goes for cost. In this sweep, the judging panel often costs several times as much as the writer.

We break down latency and cost in two ways.

- **By operation.** How much time and money go into generation, judging, and repair?
- **By part.** Which parts are slow or expensive? This helps us decide where to try a different writer model or a cheaper judging panel for a part where the judges usually agree.

In [6]:
CALLS = pd.DataFrame([{**c.__dict__, "run": r.run_id, "experiment": r.experiment}
                      for r in RUNS for c in r.calls])

if ANONYMISE:
    CALLS = CALLS.assign(model="(model)", cost=float("nan"))

by_kind = (CALLS.groupby("kind")
           .agg(calls=("label", "size"), seconds=("seconds", "sum"),
                tokens_in=("tokens_in", "sum"), tokens_out=("tokens_out", "sum"),
                usd=("cost", "sum"))
           .sort_values("usd", ascending=False).round(5))
display(by_kind)

by_part = (CALLS[CALLS["part"] != ""].groupby(["part", "kind"])
           .agg(calls=("label", "size"), seconds=("seconds", "sum"),
                usd=("cost", "sum")).round(5))
display(by_part)

if not ANONYMISE and by_kind["usd"].sum():
    share = by_kind["usd"] / by_kind["usd"].sum()
    for kind, pct in share.items():
        print(f"   {kind:<10}{pct:>6.0%} of spend")
    print("\nA panel is not free. Three judges reading a rubric for every part, "
          "every round,\nis the single largest line in most compiled-skill "
          "pipelines — which is an\nargument for cheaper judges on settled parts, "
          "not for no judges.")


,calls,seconds,tokens_in,tokens_out,usd
kind,,,,,
generate,9,37.8,5433,1006,0.01089
judge,12,30.4,15600,2624,0.00902
repair,1,4.8,1010,96,0.00181


calls  seconds      usd
part       kind                             
highlights generate      3     12.1  0.00399
           judge         6     15.9  0.00485
           repair        1      4.8  0.00181
summary    generate      3      5.5  0.00043
           judge         6     14.5  0.00417

   generate     50% of spend
   judge        42% of spend
   repair        8% of spend

A panel is not free. Three judges reading a rubric for every part, every round,
is the single largest line in most compiled-skill pipelines — which is an
argument for cheaper judges on settled parts, not for no judges.


## 5. Comparing configurations

We use the traces from both configurations to compare how they handled the same cases. The table shows latency and cost by part where the traces give us that breakdown.

We calculate latency and cost summaries from runs that produced a document and count failed runs separately. A run can fail early, and including it in the latency median can make the configuration look faster.

With only a few runs per configuration, use the table to see how the comparison works. To draw conclusions, run the same set of cases repeatedly under both configurations. That’s what the sweep is for.

When a single call generates the whole document, the trace records one generation time and one cost for that call. We leave the per-part generation cells blank and report the call totals below the table. Generating each part separately gives us the measurements we need to see which parts drive latency and cost.

In [7]:
def per_experiment(runs: Sequence[Run]) -> pd.DataFrame:
    rows = []
    for experiment in sorted({r.experiment for r in runs}):
        group = [r for r in runs if r.experiment == experiment]
        done = [r for r in group if r.ready]
        rows.append({
            "experiment": experiment,
            "runs": len(group),
            "not ready": len(group) - len(done),
            "median s": round(statistics.median([r.seconds for r in done]), 1) if done else None,
            "median USD": round(statistics.median([r.cost for r in done]), 5) if done else None,
            "calls/run": round(statistics.mean([len(r.calls) for r in group]), 1),
            "tokens in": sum(c.tokens_in for r in group for c in r.calls),
            "repairs": sum(1 for r in group for c in r.calls if c.kind == "repair"),
        })
    return pd.DataFrame(rows).set_index("experiment")


display(per_experiment(RUNS))

# The same comparison, per part: where a configuration actually differs.
rows = []
for r in RUNS:
    for part, prov in r.provenance.items():
        part_calls = [c for c in r.calls if c.part == part and c.kind == "generate"]
        # None, not 0.0: a whole-document run generated this part in a call that
        # also produced every other part, so there is no per-part figure to give.
        rows.append({"experiment": r.experiment, "part": part,
                     "model": "(model)" if ANONYMISE else prov.get("model", ""),
                     "status": prov.get("status", ""), "score": prov.get("score"),
                     "attempts": prov.get("attempts", 0),
                     "gen seconds": round(max(c.seconds for c in part_calls), 2)
                                    if part_calls else None,
                     "gen USD": round(sum(c.cost for c in part_calls), 5)
                                if part_calls else None})
PARTS = pd.DataFrame(rows)
display(PARTS.pivot_table(index="part", columns="experiment",
                          values=["gen seconds", "gen USD", "score"],
                          aggfunc="median", dropna=False))
display(PARTS)

shared = CALLS[(CALLS["kind"] == "generate") & (CALLS["part"] == "")]
if not shared.empty:
    print("\ngeneration that serves every part, so it appears in no part's row:")
    for experiment, g in shared.groupby("experiment"):
        usd = "" if ANONYMISE else f", ${g['cost'].sum():.5f}"
        print(f"   {experiment:<18}{len(g)} call(s), {g['seconds'].max():.1f}s "
              f"longest{usd}")


,runs,not ready,median s,median USD,calls/run,tokens in,repairs
experiment,,,,,,,
split+routed,1,0,7.9,0.00893,12,10380,0
whole document,1,0,11.4,0.01279,10,11663,1


gen USD                 gen seconds                 \
experiment split+routed whole document split+routed whole document   
part                                                                 
highlights      0.00399            NaN          4.2            NaN   
summary         0.00043            NaN          2.0            NaN   

                  score                 
experiment split+routed whole document  
part                                    
highlights         0.81           0.66  
summary            0.78           0.72

,experiment,part,model,status,score,attempts,gen seconds,gen USD
0,whole document,summary,openai/gpt-5.6-luna,ok,0.72,0,NaN,NaN
1,whole document,highlights,openai/gpt-5.6-luna,repaired,0.66,1,NaN,NaN
2,split+routed,summary,openai/gpt-5.4-mini,ok,0.78,0,2.0,0.00043
3,split+routed,highlights,openai/gpt-5.6-luna,ok,0.81,0,4.2,0.00399



generation that serves every part, so it appears in no part's row:
   whole document    3 call(s), 7.1s longest, $0.00647


## 6. Reviewing the documents

We rebuild each document from its parts and group the versions from the same experiment so we can read them side by side.

Each version shows the model, surface digest, run status, and score. If a reviewer flags a sentence, we can use the surface digest to identify the instructions that were used to generate it.

Set `ANONYMISE = True` to hide model names and costs in the output below. This lets reviewers compare the text without knowing which model wrote each version or how much it cost.

In [8]:
SECTION_ORDER = ["summary", "highlights"]


def as_lines(value) -> List[str]:
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    return [line.strip() for line in str(value or "").split("\n") if line.strip()]


def render(run: Run, anonymise: bool = None) -> str:
    anonymise = ANONYMISE if anonymise is None else anonymise
    out = [f"## {run.experiment}", ""]
    if not anonymise:
        out.append(f"*run `{run.run_id}` · {run.skill} · {run.seconds:.1f}s · "
                   f"${run.cost:.4f} · ready={run.ready}*")
        out.append("")
    for part in SECTION_ORDER + [p for p in run.parts if p not in SECTION_ORDER]:
        if part not in run.parts or part == "breaking":
            continue
        lines = as_lines(run.parts[part])
        if not lines:
            continue
        out.append(f"**{part.title()}**")
        out += [f"- {x}" for x in lines] if len(lines) > 1 else lines
        if not anonymise and part in run.provenance:
            p = run.provenance[part]
            out.append(f"<sub>{p.get('model','?')} · surface {p.get('surface_digest','?')} · "
                       f"{p.get('status','?')} · score {p.get('score','?')} · "
                       f"{p.get('attempts',0)} repair(s)</sub>")
        out.append("")
    if run.parts.get("breaking"):
        out += ["> This release contains a breaking change.", ""]
    return "\n".join(out)


def comparison_markdown(runs: Sequence[Run], anonymise: bool = None) -> str:
    """Every run, grouped by what it was generated from, configurations adjacent."""
    by_skill: Dict[str, List[Run]] = defaultdict(list)
    for r in runs:
        by_skill[r.skill or "unknown"].append(r)
    out = ["# Release notes, by configuration", ""]
    for skill, group in sorted(by_skill.items()):
        out += [f"# {skill}", ""]
        for r in sorted(group, key=lambda r: r.experiment):
            out.append(render(r, anonymise))
        out += ["---", ""]
    return "\n".join(out)


from IPython.display import Markdown, display
display(Markdown(comparison_markdown(RUNS)))


# Release notes, by configuration

# formal-developer

## split+routed

*run `sample-split` · formal-developer · 7.9s · $0.0089 · ready=True*

**Summary**
Version 2.4.0 rebuilds the CSV exporter; `/v1/export` now returns a job id rather than the file.
<sub>openai/gpt-5.4-mini · surface 77aa11bb22cc · ok · score 0.78 · 0 repair(s)</sub>

**Highlights**
- Improved export throughput, approximately two times
- Reduced peak memory during export by approximately 40 percent
- Corrected escaping of quotation marks inside CSV fields
- Revised `/v1/export` response: `202` with a job id, polled at `/v1/export/{id}`
<sub>openai/gpt-5.6-luna · surface 9f8e7d6c5b4a · ok · score 0.81 · 0 repair(s)</sub>

> This release contains a breaking change.

## whole document

*run `sample-whole` · formal-developer · 11.4s · $0.0128 · ready=True*

**Summary**
Version 2.4.0 rebuilds the CSV exporter and changes `/v1/export` to return a job id instead of the file.
<sub>openai/gpt-5.6-luna · surface a1b2c3d4e5f6 · ok · score 0.72 · 0 repair(s)</sub>

**Highlights**
- Improved export throughput, approximately two times
- Reduced peak memory during export by approximately 40 percent
- Revised `/v1/export` response: `202` with a job id, polled at `/v1/export/{id}`
<sub>openai/gpt-5.6-luna · surface a1b2c3d4e5f6 · repaired · score 0.66 · 1 repair(s)</sub>

> This release contains a breaking change.

---


## 7. Exporting the documents and tables

We write one Markdown file per run, a Word document with the versions for comparison, and the tables as CSV files to `export/`. The notebook bundles these files into a ZIP and starts the download automatically in Colab. We can share the ZIP with reviewers so they can work outside Langfuse.

In Word, a subject-matter expert can leave a comment on a specific sentence or suggest a revision with Track Changes. When we get the document back, we can see exactly which wording each comment refers to.

In [9]:
from docx import Document
from docx.shared import Pt

FOLDER = Path("export")
FOLDER.mkdir(exist_ok=True)


def safe(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_-]+", "_", name).strip("_")[:80]


def as_word(run: Run, path: Path, anonymise: bool = None) -> None:
    anonymise = ANONYMISE if anonymise is None else anonymise
    doc = Document()
    doc.styles["Normal"].font.name = "Calibri"
    doc.styles["Normal"].font.size = Pt(11)
    doc.add_heading(f"Release notes — {run.experiment}", level=1)
    if not anonymise:
        doc.add_paragraph(f"{run.skill} · run {run.run_id} · {run.seconds:.1f}s · "
                          f"${run.cost:.4f}")
    for part in SECTION_ORDER + [p for p in run.parts if p not in SECTION_ORDER]:
        if part not in run.parts or part == "breaking":
            continue
        lines = as_lines(run.parts[part])
        if not lines:
            continue
        doc.add_heading(part.title(), level=2)
        for line in lines:
            doc.add_paragraph(line, style="List Bullet" if len(lines) > 1 else None)
        if not anonymise and part in run.provenance:
            p = run.provenance[part]
            doc.add_paragraph(f"{p.get('model','?')} · surface {p.get('surface_digest','?')} "
                              f"· {p.get('status','?')}").italic = True
    doc.save(path)


for run in RUNS:
    stem = safe(f"{run.experiment}_{run.run_id}")
    (FOLDER / f"{stem}.md").write_text(render(run), encoding="utf-8")
    as_word(run, FOLDER / f"{stem}.docx")

(FOLDER / "comparison.md").write_text(comparison_markdown(RUNS), encoding="utf-8")
(FOLDER / "comparison_blind.md").write_text(comparison_markdown(RUNS, anonymise=True),
                                            encoding="utf-8")
CALLS.to_csv(FOLDER / "calls.csv", index=False)
PARTS.to_csv(FOLDER / "parts.csv", index=False)
per_experiment(RUNS).to_csv(FOLDER / "experiments.csv")

archive = Path("release_notes_export.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(FOLDER.iterdir()):
        z.write(f, f.name)
print(f"{len(list(FOLDER.iterdir()))} file(s) in {FOLDER}/ and {archive}")

try:                                    # Colab downloads it; locally, skip
    from google.colab import files
    files.download(str(archive))
except Exception:
    pass


9 file(s) in export/ and release_notes_export.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Turning failures into regression cases

We save cases where a part needed repair, no judge accepted a part, or the final document failed to assemble. For each case, we record the input, the affected part if there is one, what went wrong, and the configuration that produced the run.

These cases come from actual runs, so they can capture failure modes we wouldn’t have thought to include when writing test cases by hand. A run that recovered after repair is worth keeping too. We can use it to check whether a later version gets that part right on the first attempt.

In the evaluation-pipelines notebook, we load these cases into a Langfuse dataset and score them with the same metrics before and after a skill change.

In [10]:
def failures(runs: Sequence[Run]) -> pd.DataFrame:
    rows = []
    for r in runs:
        for part, prov in r.provenance.items():
            status, attempts = prov.get("status", ""), prov.get("attempts", 0)
            if status in ("ok",) and not attempts and r.ready:
                continue
            rows.append({"run": r.run_id, "experiment": r.experiment, "skill": r.skill,
                         "part": part, "status": status, "attempts": attempts,
                         "score": prov.get("score"),
                         "model": "(model)" if ANONYMISE else prov.get("model", ""),
                         "surface_digest": prov.get("surface_digest", ""),
                         "text": str(r.parts.get(part))[:160]})
    return pd.DataFrame(rows)


CASES = failures(RUNS)
print(f"{len(CASES)} case(s) worth keeping "
      f"(a repair, a rejection, or a document that never assembled)\n")
display(CASES)

(FOLDER / "regression_cases.json").write_text(
    CASES.to_json(orient="records", indent=1), encoding="utf-8")
print(f"written to {FOLDER / 'regression_cases.json'} — the input for a dataset")


1 case(s) worth keeping (a repair, a rejection, or a document that never assembled)



,run,experiment,skill,part,status,attempts,score,model,surface_digest,text
0,sample-whole,whole document,formal-developer,highlights,repaired,1,0.66,openai/gpt-5.6-luna,a1b2c3d4e5f6,"['Improved export throughput, approximately tw..."


written to export/regression_cases.json — the input for a dataset


## 9. Using the notebooks together

The compiler notebook controls which rules each call receives and records what happened during the run. This notebook uses those traces to compare configurations, review the generated documents, and collect regression cases.

| | Compiler notebook | This notebook |
|---|---|---|
| What we look at | Which rules each call receives | How the configurations compare on the same cases |
| What we get | Generated documents and execution traces | Latency and cost tables, documents for review, and regression cases |

1. **Use one session ID per configuration in the sweep.** This lets the notebook group runs by configuration when it builds the comparison.
2. **Send the anonymised comparison to the rule owner.** They can review the text without seeing which model generated each version or how much it cost.
3. **Keep cases that failed or needed repair.** Add them to the regression set so future changes get tested against issues we’ve seen on actual inputs.
4. **Rerun the comparison after each skill change.** The compiled skill lets us reproduce which rules each call received. Running the same cases again shows what changed in the output, how often we needed repair, and what happened to latency and cost.